# Imports

In [9]:
import pickle # Load refs and annotations
import json
import os
import pandas as pd
import numpy as np
import pprint
import json
import cv2
import random

from typing import Any, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from torch.utils.tensorboard import SummaryWriter

import torchvision
import torchvision.transforms as transforms

import torchmetrics

import pytorch_lightning as pl
from pytorch_lightning.utilities.types import STEP_OUTPUT

from transformers import T5Tokenizer, T5ForConditionalGeneration
from transformers import CLIPProcessor, CLIPModel

from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import clip
from ultralytics import YOLO
from PIL import Image, ImageDraw

from ipywidgets import FloatProgress

# Setup

We just specify the best backend available for operations

In [10]:
if torch.backends.mps.is_available():
    print("MPS backend is available.")
    device = torch.device('mps')
elif torch.cuda.is_available():
    print("CUDA backend is available.")
    device = torch.device('cuda')
else:
    print("Neither CUDA or MPS backend are available. Resorting to CPU")
    device = torch.device('cpu')

CUDA backend is available.


In [11]:
device

device(type='cuda')

In [12]:
zeroshotYolo = True

As per request of the assignment we resorted to clip based on a ResNet50 to be the core of our architecture

In [13]:
clip_model, clip_preprocess = clip.load("RN50", device=device)

compute_iou function is just an helper function for calculating the Intersection over Union of two bboxes. 
The bboxes need to be served as a list of coordinates served as $[x_1,y_1,x_2,y_2]$

In [27]:
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_area = max(0, x2 - x1 + 1) * max(0, y2 - y1 + 1) # +1 to avoid max(0,0) therefore avoiding 

    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = box1_area + box2_area - intersection_area

    return intersection_area / union_area

The LossMeter class has been created to help keeping track of the different values during training stages for all the different experiments

In [14]:
class LossMeter:
    def __init__(self, name="Default"):
        self.name = name
        self.losses = []
        self.reset()

    def reset(self):
        self.avg, self.sum, self.count = 0,0,0
        self.max_iou = 0

    def update(self, val, count=1):
        self.count += count
        self.sum += val * count
        self.avg = self.sum / self.count
        self.losses.append([self.count, self.sum, self.avg])

    def __repr__(self):
        text = f"{self.name}: {self.avg:.4f}"
        return text
    
    def localization_accuracy(self, candidates,gt):
        for bbox_e in candidates:
            pred = compute_iou(gt,bbox_e[1])
            if pred > self.max_iou:
                self.max_iou = pred
    
    

def get_lr(optimizer):
    for param_group in optimizer.param_groups:
        return param_group["lr"]

# Data

Here we recover data from the given dataset by mapping values retrieved from the pickle file with the one found in the json

In [16]:
with open("../refcocog/annotations/refs(umd).p", "rb") as fp:
  refs = pickle.load(fp)

# 'annotations' will be a dict object mapping the 'annotation_id' to the 'bbox' to make search faster
with open("../refcocog/annotations/instances.json", "rb") as fp:
  data = json.load(fp)
  annotations = dict(sorted({ann["id"]: ann["bbox"] for ann in data["annotations"]}.items()))

In [17]:
def getcaption(elem):
    li = []
    for e in elem["sentences"]:
        li.append(e['raw'])
    return li

Here we create the dataset class for our project

In [18]:
class RefCOCOG(Dataset):
    """
    Args:
        The dataset will be the raw data wothput any tipe of preprocessing
        {
            'file_name': 
            'caption':
            'ann_id': needed to extract the relative bbox from the .json file
            'bbox': values are set like following:
                - x 
                - y
                - width 
                - height
        }
    """
    def __init__(self, refs, annotations, split="train"):

        dataset = list()

        for elem in [d for d in refs if d["split"]==split]:
            file_name = os.path.join("../refcocog/images/", f'{"_".join(elem["file_name"].split("_")[:3])}.jpg')
            sentences = elem['sentences']
            raws = []
            for i in sentences:
                raws.append(i['raw'])
            bbox = annotations[elem['ann_id']]
            for i in raws:
                dataset.append({"file_name":file_name,"raw":i,"bbox":bbox})

        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]

    def __call__(self, idx):
        print(json.dumps(self.dataset[idx], indent=4))


And here we set up some functions for the dataloader. 
- in pad_image we just pad every image from the dataset to a set fixed dimension of 640x640
- the collate function just prepares the data by preprocessing the dataset and creating a dataloader

In [19]:
def pad_image(image):
    """
    Performs bottom-right padding of the original image to 640x640 (max size of images in the dataset).
    Bottom-right padding prevents corruption of bounding boxes.

    ### Arguments
    image: a PIL.Image to transform
    """
    padded_width, padded_height = 640, 640
    original_height, original_width = image.shape[:2]
    bottom_padding = padded_height - original_height
    right_padding = padded_width - original_width
    top_padding = 0
    left_padding = 0
    
    padded_image = cv2.copyMakeBorder(image, top_padding, bottom_padding, left_padding, right_padding, cv2.BORDER_CONSTANT, value=[0, 0, 0])

    return padded_image 

def collate_fn(batch):
    #print("Actual batch is made of : " + batch)
    images = []
    data = {}

    #Stores all images in a list
    for sample in batch:
        image = cv2.imread(sample["file_name"], 3)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = pad_image(image=image)

        y_ = image.shape[0]
        x_ = image.shape[1]


        images.append(transform(image))

        data['raw'] = sample['raw']
        x,y,z,c = sample['bbox'][0:4]
        y1=y 
        x1=x 
        y2=(y + c)
        x2=(x + z)

        data['bbox'] = [x,y,x2,y2]
        data['filename'] = sample["file_name"]
            
    images = torch.stack(images, dim=0)
    """
    for key in batch[0].keys():
        #if key != "file_name":
        #    data[key] = [sample[key] for sample in batch]
        data[key] = [sample[key] for sample in batch]
        if( key == 'bbox'):
            x,y,z,c = sample[key][0:4]
            y1=y 
            x1=x 
            y2=(y + z)
            x2=(x + c)
    """
    return images, data

transform = transforms.Compose([
    transforms.ToTensor(),
])

In [20]:
# create dataset and dataloader
dataset_test = RefCOCOG(refs, annotations, split="test")
dataset_valid = RefCOCOG(refs, annotations, split="train")
#print(dataset_valid[1]['file_name'])
print("---------------------------------------------------")
#plt.imshow(Image.open(dataset[2]["file_name"]))
dataloader_test = DataLoader(dataset_test, batch_size=16, collate_fn=collate_fn, num_workers=0)
dataloader_valid = DataLoader(dataset_test, batch_size=16, collate_fn=collate_fn, num_workers=0)


---------------------------------------------------


# Zero Shot Yolo - CLIP (ResNet50)

## Architecture

As the name of the model suggests this is a straightforward implementation of a Yolov8 model with the CLIP architecture.
We took the nano version just for the sake of implementation since it was the only one that could run on our local machine

In [21]:
##########################################################
# YolottoClip - Just for Zero-Shotting the dataset
##########################################################

class YolottoClip():
    def __init__(self, clip_model, clip_preprocess):

        self.yolo = YOLO("yolov8n.pt")
        self.clip_model, self.clip_preprocess = clip_model, clip_preprocess

    def infer_bboxes(self, image_path):
        results = self.yolo(image_path, verbose=False)
        bboxes = results[0].boxes.xyxy
        return bboxes

    def encode_image(self, image):
        # Load and preprocess the image using CLIP preprocess function
        image = self.clip_preprocess(image).unsqueeze(0).to(self.device)

        # Encode the image using the CLIP model
        with torch.no_grad():
            image_features = self.clip_model.encode_image(image)

        return image_features

    def encode_text(self, text):
        # Encode the text using the CLIP model
        text = clip.tokenize(text).to(self.device)
        with torch.no_grad():
            text_features = self.clip_model.encode_text(text)

        return text_features
    
    def forward(self, image, text):

        text = clip.tokenize(text).to(device)

        image_features = self.encode_image(image)
        text_features = self.encode_text(text)

        # normalized features
        image_features = image_features / image_features.norm(dim=1, keepdim=True)
        text_features = text_features / text_features.norm(dim=1, keepdim=True)

        # cosine similarity as logits
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image_features @ text_features.t()
        logits_per_text = logits_per_image.t()

        # shape = [global_batch_size, global_batch_size]
        return logits_per_image, logits_per_text
    
    def calculate_best_bbox(self, image_path, caption, device):
        text = clip.tokenize(caption).to(device)
        best_score = 0
        best_bbox = None
        candidates = []

        for bbox in self.infer_bboxes(image_path):
            temp = cv2.imread(image_path)
            image = np.zeros((temp.shape[0], temp.shape[1], temp.shape[2]), dtype=np.uint8)
            image[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])] = temp[int(bbox[1]):int(bbox[3]), int(bbox[0]):int(bbox[2])]
            image = self.clip_preprocess(Image.fromarray(image)).unsqueeze(0).to(device)

            with torch.no_grad():
                logits_per_image, logits_per_text = self.forward(image, text)
                matching_score = logits_per_text.cpu().numpy()[0]

            candidates.append((matching_score,bbox))

            if matching_score > best_score:
                best_score = matching_score
                best_bbox = bbox

        return best_score, best_bbox, candidates
    

In [22]:
ZeroShotter = YolottoClip(clip_model,clip_preprocess)

100%|██████████| 6.25M/6.25M [00:00<00:00, 7.00MB/s]


## Test on a Single Image

In [23]:
img = 'zidane.jpg'
caption = 'bald man with a tie'

In [24]:
yolo = YOLO("../checkpoints/yolov8n.pt")

In [25]:
NUM_EPOCHS = 1
count = 0
loss_meter = LossMeter()
iou_threshold = 0.5

running_loc_acc = 0.0
correct_bboxes = 0
overall = 0
running_ss = 0
semantic_similarity = 0


In [26]:
for epoch in range(NUM_EPOCHS):
    loop = tqdm(dataloader_test, position=0, leave=True)
    for iter in enumerate(loop):
        try:
            data = iter[1][1]

            raw_text = data['raw']
            bbox_gt = data['bbox']
            filename = data['filename']
            # since confidence is directly how much bbox and text "resembles" each other 
            confidence, bbox_e, candidates = ZeroShotter.calculate_best_bbox(filename,raw_text,device)

            bbox_e = [int(bbox_e[0]), int(bbox_e[1]), int(bbox_e[2]), int(bbox_e[3])]

            max_iou = 0.0
            for bbox_e in candidates:
                pred = compute_iou(bbox_gt,bbox_e[1])
                if pred > max_iou:
                    max_iou = pred
                #print(pred)

            running_loc_acc += max_iou
                

            if(pred > iou_threshold):
                #compare_candidate_bbox(filename,bbox_e, bbox_gt)
                correct_bboxes += 1
                overall += 1
                running_ga = correct_bboxes / overall
            else:
                overall += 1
                running_ga = correct_bboxes / overall

            running_ss += confidence[0] / 100
            semantic_similarity = running_ss / overall
            loc_acc = running_loc_acc / overall
        except:
            print(filename)

        loop.set_description(f"Localization accuracy = {loc_acc}, Grounding Accuracy = {running_ga}, Semantic Similarity = {semantic_similarity}")

  0%|          | 0/601 [00:00<?, ?it/s]

(0, (tensor([[[[0.7647, 0.7490, 0.7451,  ..., 0.8275, 0.8275, 0.8078],
          [0.7922, 0.7961, 0.7922,  ..., 0.8235, 0.8275, 0.8235],
          [0.7882, 0.7922, 0.7882,  ..., 0.8314, 0.8235, 0.8118],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

         [[0.7686, 0.7529, 0.7490,  ..., 0.8275, 0.8275, 0.8078],
          [0.7922, 0.7961, 0.7922,  ..., 0.8235, 0.8275, 0.8235],
          [0.7882, 0.7922, 0.7882,  ..., 0.8314, 0.8235, 0.8157],
          ...,
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

         [[0.7765, 0.7608, 0.7569,  ..., 0.8275, 0.8275, 0.8000],
          [0.8000, 0.8039, 0.8000,  ..., 0.8235, 0.8196, 0.8157],
          [0.7961, 0.8000, 0.7961,  .

NameError: name 'loc_acc' is not defined